# 01 — Whitening

**Stage 1.** Load van Hateren linear `.iml` images, extract patches, and apply the
from-scratch frequency-domain whitening filter `R(f) = f · exp(−(f/f₀)⁴)`.

**Deliverable:** raw vs whitened patch power spectra, plus the one `f₀` sensitivity
sweep. `f₀` is locked at **0.4 cycles/pixel** (Olshausen sparsenet convention) — not
tuned to prettify downstream filters.

Pipeline: load → log-luminance → whiten the **whole image** → extract patches.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sparse_coding.data_loading import load_images, preprocess_image, extract_patches
from sparse_coding.whitening import whiten_image, radially_averaged_power_spectrum

F0 = 0.4            # locked whitening cutoff, cycles/pixel
PATCH_SIZE = 16
N_PATCHES = 500
SEED = 0
DATA_DIR = Path("../data")

## Load an image (or a synthetic fallback)

If no `.iml` files are in `data/` yet, we synthesize a 1/f natural-image-like field so
the notebook still runs end-to-end. Download the van Hateren linear set into `data/`
(see README → Data sources) to run on real images.

In [ ]:
iml_files = sorted(DATA_DIR.glob("*.iml"))
if iml_files:
    raw = load_images(DATA_DIR)[0]
    USING_SYNTHETIC = False
    print(f"Loaded {iml_files[0].name}  shape={raw.shape}")
else:
    # Synthetic 1/f fallback so the notebook runs before data is downloaded.
    rng = np.random.default_rng(SEED)
    N = 512
    fy = np.fft.fftfreq(N)[:, None]
    fx = np.fft.fftfreq(N)[None, :]
    rho = np.sqrt(fx**2 + fy**2)
    amp = np.zeros_like(rho)
    np.divide(1.0, rho, out=amp, where=rho > 0)
    raw = np.real(np.fft.ifft2(np.fft.fft2(rng.standard_normal((N, N))) * amp))
    raw = raw - raw.min() + 1.0  # make positive so log-luminance is defined
    USING_SYNTHETIC = True
    print("No .iml found in data/ — using SYNTHETIC 1/f fallback.")

## Preprocess and whiten the whole image

In [ ]:
# log-luminance only for real linear images; the synthetic field is already ~Gaussian.
img = preprocess_image(raw, log_luminance=not USING_SYNTHETIC)
img_white = whiten_image(img, f0=F0)

fig, ax = plt.subplots(1, 2, figsize=(11, 5))
src = "synthetic 1/f" if USING_SYNTHETIC else "log-luminance"
ax[0].imshow(img, cmap="gray"); ax[0].set_title(f"input ({src})"); ax[0].axis("off")
ax[1].imshow(img_white, cmap="gray"); ax[1].set_title(f"whitened (f0={F0})"); ax[1].axis("off")
plt.tight_layout(); plt.show()

## Extract matched patches

Identical seeds → identical patch locations, so raw and whitened patches are compared at
the same positions.

In [ ]:
raw_patches = extract_patches([img], N_PATCHES, PATCH_SIZE, rng=np.random.default_rng(SEED))
white_patches = extract_patches([img_white], N_PATCHES, PATCH_SIZE, rng=np.random.default_rng(SEED))

fig, ax = plt.subplots(1, 2, figsize=(6, 3))
ax[0].imshow(raw_patches[0], cmap="gray"); ax[0].set_title("raw patch"); ax[0].axis("off")
ax[1].imshow(white_patches[0], cmap="gray"); ax[1].set_title("whitened patch"); ax[1].axis("off")
plt.tight_layout(); plt.show()

## Raw vs whitened patch power spectra

The stage deliverable, measured on the patches that actually feed dictionary learning.
Raw patches fall as ~1/f; whitening flattens the spectrum through the passband and rolls
off past `f₀`. Coarser than the whole-image view above (only ~8 bins at 16px), which is
exactly why whitening is applied to the whole image rather than per patch.

In [ ]:
fI_raw, pI_raw = radially_averaged_power_spectrum(img)
fI_wht, pI_wht = radially_averaged_power_spectrum(img_white)

plt.figure(figsize=(6, 5))
plt.loglog(fI_raw, pI_raw, label="raw")
plt.loglog(fI_wht, pI_wht, label="whitened")
plt.axvline(F0, color="k", ls=":", alpha=0.5, label=f"f0={F0}")
plt.xlabel("spatial frequency (cyc/pixel)"); plt.ylabel("radially-averaged power")
plt.title("Whole-image power spectra: raw vs whitened"); plt.legend()
plt.grid(True, which="both", alpha=0.3); plt.show()

## Raw vs whitened patch power spectra

The headline figure. Raw natural-image patches fall as ~1/f (a downward slope on log-log
axes); whitening flattens the spectrum through the passband and rolls off past `f₀`.

In [ ]:
def mean_patch_spectrum(patches):
    """Average the radially-averaged power spectrum over patches (same size → same bins)."""
    freqs, acc = None, None
    for p in patches:
        f, pw = radially_averaged_power_spectrum(p)
        if freqs is None:
            freqs, acc = f, np.zeros_like(pw)
        acc += pw
    return freqs, acc / len(patches)

f_raw, p_raw = mean_patch_spectrum(raw_patches)
f_wht, p_wht = mean_patch_spectrum(white_patches)

plt.figure(figsize=(6, 5))
plt.loglog(f_raw, p_raw, "o-", label="raw")
plt.loglog(f_wht, p_wht, "o-", label="whitened")
plt.axvline(F0, color="k", ls=":", alpha=0.5, label=f"f0={F0}")
plt.xlabel("spatial frequency (cyc/pixel)"); plt.ylabel("radially-averaged power")
plt.title("Patch power spectra: raw vs whitened"); plt.legend()
plt.grid(True, which="both", alpha=0.3); plt.show()

## f₀ sensitivity sweep

`f₀` is **locked at 0.4** — this sweep is shown for honesty (how the whitened spectrum
moves with the cutoff), not to pick a prettier value after the fact.

In [ ]:
plt.figure(figsize=(6, 5))
for f0 in [0.2, 0.3, 0.4, 0.5]:
    wimg = whiten_image(img, f0=f0)
    wp = extract_patches([wimg], N_PATCHES, PATCH_SIZE, rng=np.random.default_rng(SEED))
    f, pw = mean_patch_spectrum(wp)
    style = "-" if f0 == F0 else "--"
    lw = 2.5 if f0 == F0 else 1.0
    plt.loglog(f, pw, style, lw=lw, label=f"f0={f0}" + (" (locked)" if f0 == F0 else ""))
plt.loglog(f_raw, p_raw, "k:", alpha=0.5, label="raw")
plt.xlabel("spatial frequency (cyc/pixel)"); plt.ylabel("radially-averaged power")
plt.title("f0 sensitivity sweep"); plt.legend()
plt.grid(True, which="both", alpha=0.3); plt.show()